# Network formation with a single player fixed effect

A directed network formation game with reciprocity. Each player $t \in \{1, \dots, T\}$ chooses an out-link vector $d_t = (d_{ts})_{s \ne t} \in \{0, 1\}^{T-1}$ given the other players' choices $d_{-t}$. The full action profile is $d = (d_{ts})_{t \ne s} \in \{0, 1\}^M$ with $M = T(T-1)$ directed dyads.

The per-player payoff is

$$v_t(d_t \mid d_{-t}) = \sum_{s \ne t} d_{ts} \big( \alpha_t + \alpha_s + r_{(t,s)}^\top \beta + \nu_{(t,s)} \big) + \gamma \sum_{s \ne t} d_{ts}\, d_{st}.$$

A pure-strategy Nash equilibrium is a profile $d^*$ such that $d_t^* \in \arg\max_{d_t} v_t(d_t \mid d^*_{-t})$ for every $t$. The game admits the potential

$$V(d; \alpha, \beta, \gamma , \nu) = \sum_{t \ne s} d_{ts} \big( \alpha_t + \alpha_s + r_{(t,s)}^\top \beta + \nu_{(t,s)} \big) + \gamma \sum_{t < s} d_{ts}\, d_{st},$$

which satisfies $V(d_t, d_{-t}) - V(d_t', d_{-t}) = v_t(d_t \mid d_{-t}) - v_t(d_t' \mid d_{-t})$ for every $t$. The reciprocity sum runs over unordered pairs in $V$ to avoid double-counting when payoffs are aggregated. Any global maximizer of $V$ is a Nash equilibrium (Monderer and Shapley, 1996).


In [1]:
from dataclasses import dataclass

import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import breadth_first_order, maximum_flow

import combrum as cb

T = 50
M = T * (T - 1)
K_BETA = 5
S = 1000
DESIGN_SEED = 20260424
SHOCK_SEED = 4000
ACTIVITY_LEVEL = "iterations"
BETA_TRUE = np.array([0.8, 0.5, 1.0, 0.3, 0.4])
GAMMA_TRUE = 0.5
FE_MEAN = -1.0
FE_SD = 0.5

parameters = cb.Parameters(
    {
        "alpha": (-8.0, 8.0, T),
        "beta": (-8.0, 8.0, K_BETA),
        "gamma": (0.0, 8.0, 1),
    }
)
print(f"T={T}  M={M}  S={S}  K={parameters.K}")


T=50  M=2450  S=1000  K=56


## Primitives

A **dyad** is an ordered pair $(t, s)$ with $t \ne s$. There are $M = T(T-1)$ dyads.

Per player $t$, draw:

- two binary group labels $g_t^{(1)}, g_t^{(2)} \in \{0, 1\}$
- two continuous attributes $a_t \sim U(0, 1)$ and $b_t \sim N(0, 1)$
- a continuous attribute $z_t \sim N(0, 1)$

Per dyad $(t, s)$, the covariate vector $r_{(t,s)} \in \mathbb{R}^5$ has entries:

1. $\mathbb{1}\{g_t^{(1)} = g_s^{(1)}\}$
2. $\mathbb{1}\{g_t^{(2)} = g_s^{(2)}\}$
3. $-|a_t - a_s|$
4. $-|b_t - b_s|$
5. $z_t \cdot z_s$

These rows stack into $R \in \mathbb{R}^{M \times 5}$. A single player fixed-effect block enters each dyad as $\alpha_t + \alpha_s$, so the feature for player $j$ counts selected links where $j$ is either endpoint.

Reciprocity enters through $d_{ts} d_{st}$: it is one only when players $t$ and $s$ link to each other. We sum over $t<s$ so each pair appears once.

$\alpha_t \sim N(-1, 0.5)$ iid.


For $\gamma \ge 0$ the potential is supermodular and quadratic in $d$ and its argmax reduces to a min-cut problem via the Boros and Hammer transform. The observed network is one realization of $\arg\max_d V(d)$ at $\theta_{\text{true}}$. This is used in `NetworkDemandOracle` implemented below.

In [2]:

def min_cut_bundle(linear, edge_rows, edge_cols, pair):
    upper_rowsum = np.zeros(linear.size)
    np.add.at(upper_rowsum, edge_rows, pair)
    net = -linear - upper_rowsum
    magnitude = max(np.abs(net).max(initial=0.0), pair.max(initial=0.0))
    scale = 1.0 if magnitude == 0.0 else 10.0 ** (8 - np.floor(np.log10(magnitude)))
    net_int = np.round(net * scale).astype(np.int64)
    pair_int = np.round(pair * scale).astype(np.int64)

    source, sink = 0, 1
    nodes = np.arange(linear.size) + 2
    select = net_int < 0
    present = pair_int > 0
    rows = np.concatenate([np.full(select.sum(), source), nodes[~select], edge_rows[present] + 2])
    cols = np.concatenate([nodes[select], np.full((~select).sum(), sink), edge_cols[present] + 2])
    vals = np.concatenate([-net_int[select], net_int[~select], pair_int[present]])
    cap = csr_matrix((vals, (rows, cols)), shape=(linear.size + 2, linear.size + 2), dtype=np.int64)
    flow = maximum_flow(cap, source, sink).flow
    residual = (cap - flow).tocsr()
    residual.data[residual.data <= 0] = 0
    residual.eliminate_zeros()
    _, pred = breadth_first_order(residual, i_start=source, directed=True, return_predecessors=True)
    reachable = pred != -9999
    reachable[source] = True
    return reachable[2:]


## Simulate the observed network at $\theta_{\text{true}}$

In [3]:

@dataclass(frozen=True)
class NetworkDesign:
    r: np.ndarray
    snd: np.ndarray
    rcv: np.ndarray
    edge_rows: np.ndarray
    edge_cols: np.ndarray
    shocks: np.ndarray
    observed: np.ndarray
    theta_true: np.ndarray


def build_design():
    rng = np.random.default_rng(DESIGN_SEED)
    snd, rcv = np.nonzero(~np.eye(T, dtype=bool))
    pair_index = np.empty((T, T), dtype=np.int64)
    pair_index[snd, rcv] = np.arange(M)
    recip = pair_index[rcv, snd]

    g1 = rng.integers(0, 2, size=T)
    g2 = rng.integers(0, 2, size=T)
    a = rng.uniform(0.0, 1.0, size=T)
    b = rng.normal(0.0, 1.0, size=T)
    z = rng.normal(0.0, 1.0, size=T)
    r = np.column_stack(
        [
            g1[snd] == g1[rcv],
            g2[snd] == g2[rcv],
            -np.abs(a[snd] - a[rcv]),
            -np.abs(b[snd] - b[rcv]),
            z[snd] * z[rcv],
        ]
    )

    alpha = rng.normal(FE_MEAN, FE_SD, size=T)
    theta_true = parameters.pack(
        {
            "alpha": alpha,
            "beta": BETA_TRUE,
            "gamma": [GAMMA_TRUE],
        }
    )
    shocks = np.random.default_rng(SHOCK_SEED).standard_normal((S, M))
    edge_rows = np.flatnonzero(np.arange(M) < recip)
    edge_cols = recip[edge_rows]
    xbeta = np.einsum("mk,k->m", r, BETA_TRUE, optimize=True)
    base = alpha[snd] + alpha[rcv] + xbeta
    observed = min_cut_bundle(base + shocks[0], edge_rows, edge_cols, np.full(edge_rows.size, GAMMA_TRUE))
    return NetworkDesign(r, snd, rcv, edge_rows, edge_cols, shocks, observed.reshape(1, M), theta_true)


design = build_design()
print(f"observed links: {design.observed.sum()} of {M}")


observed links: 240 of 2450


## Estimate via combRUM

In [4]:

class NetworkFeatures(cb.FeatureMap):
    def __init__(self, design):
        self.design = design
        cols = np.arange(M)
        ones = np.ones(M)
        sender = csr_matrix((ones, (design.snd, cols)), shape=(T, M))
        receiver = csr_matrix((ones, (design.rcv, cols)), shape=(T, M))
        self.incidence = sender + receiver

    def features_batch(
        self,
        ids,
        bundles,
        *,
        weights=None,
        aggregate=False,
    ):
        if aggregate:
            weighted_bundle = np.einsum("n,nm->m", weights, bundles, optimize=True)
            phi = np.zeros(parameters.K)
            phi[:T] = self.incidence @ weighted_bundle
            phi[T : T + K_BETA] = np.einsum(
                "m,mk->k",
                weighted_bundle,
                self.design.r,
                optimize=True,
            )
            phi[-1] = np.einsum(
                "n,n->",
                weights,
                (bundles[:, self.design.edge_rows] * bundles[:, self.design.edge_cols]).sum(axis=1),
                optimize=True,
            )
            eps = np.einsum("n,nm,nm->", weights, self.design.shocks[ids], bundles, optimize=True)
            return phi, eps

        Phi = np.zeros((bundles.shape[0], parameters.K))
        Phi[:, :T] = (self.incidence @ bundles.T).T
        Phi[:, T : T + K_BETA] = np.einsum(
            "nm,mk->nk",
            bundles,
            self.design.r,
            optimize=True,
        )
        Phi[:, -1] = (
            bundles[:, self.design.edge_rows] * bundles[:, self.design.edge_cols]
        ).sum(axis=1)
        eps = np.einsum("nm,nm->n", self.design.shocks[ids], bundles, optimize=True)
        return Phi, eps


class NetworkDemandOracle(cb.Oracle):
    def __init__(self, design):
        self.design = design

    def price_batch(self, theta, local_ids):
        values = parameters.unpack(theta)
        alpha = values["alpha"]
        beta = values["beta"]
        gamma = values["gamma"][0]
        xbeta = np.einsum("mk,k->m", self.design.r, beta, optimize=True)
        base = alpha[self.design.snd] + alpha[self.design.rcv] + xbeta
        pair = np.full(self.design.edge_rows.size, gamma)

        out = {}
        for agent_id in local_ids:
            linear = base + self.design.shocks[agent_id]
            bundle = min_cut_bundle(linear, self.design.edge_rows, self.design.edge_cols, pair)
            payoff = (
                np.sum(linear, where=bundle)
                + gamma
                * np.count_nonzero(
                    bundle[self.design.edge_rows] & bundle[self.design.edge_cols]
                )
            )
            out[agent_id] = cb.Demand.exact(bundle, payoff)
        return out


In [5]:

features = NetworkFeatures(design)
demand = NetworkDemandOracle(design)
model = cb.Model(demand, parameters, features=features, formulation=cb.NSlack)
data = cb.Data(
    observed_bundles=design.observed,
    shocks=design.shocks.reshape(1, S, M),
    observables=[0],
)

cut_policy = cb.PurgeInactive(max_age=15)
fit = cb.estimate(
    model,
    data,
    master_backend="auto",
    tolerance=1e-6,
    max_iterations=2000,
    cut_policy=cut_policy,
    activity=cb.ActivityConfig(
        label="network",
        level=ACTIVITY_LEVEL,
        stdout=True,
    ),
)


[network] row generation: N=1 S=1000 K=56 agents=1000 ranks=1 tol=1.000e-06 max_iter=2000 cuts=PurgeInactive
[network] iter         gap        dgap         obj        dobj  +cuts   cuts  priced  inexact   price  master      dt    total
[network]    0   9.873e+04           -  -3.830e+06           -   1000   1000    1000        0   0.35s   0.01s   0.36s    0.36s
[network]    1   1.641e+04  -8.233e+04  -2.051e+06   1.779e+06   1000   2000    1000        0   0.33s   0.06s   0.38s    0.75s
[network]    2   1.499e+04  -1.417e+03  -1.870e+06   1.807e+05   1000   3000    1000        0   0.33s   0.08s   0.41s    1.16s
[network]    3   6.709e+03  -8.282e+03  -1.200e+06   6.703e+05   1000   4000    1000        0   0.32s   0.13s   0.45s    1.60s
[network]    4   7.360e+03   6.505e+02  -1.013e+06   1.874e+05   1000   5000    1000        0   0.32s   0.15s   0.47s    2.08s
[network]    5   5.816e+03  -1.543e+03  -8.669e+05   1.457e+05   1000   6000    1000        0   0.32s   0.20s   0.52s    2.60s
[n

In [6]:

named = fit.theta_named()
true = parameters.unpack(design.theta_true)
alpha_hat = named["alpha"]
beta_hat = named["beta"]
gamma_hat = named["gamma"][0]
alpha_true = true["alpha"]
beta_true = true["beta"]
gamma_true = true["gamma"][0]

print(f"iterations: {fit.metadata['iterations']}   converged: {fit.metadata['converged']}   wall: {fit.runtime_seconds:.1f}s")
print(f"beta_true: {beta_true}")
print(f"beta_hat:  {beta_hat}")
print(f"gamma_true: {gamma_true:.3f}   gamma_hat: {gamma_hat:.3f}")
print(f"alpha FE correlation: {np.corrcoef(alpha_hat, alpha_true)[0, 1]:.3f}")


iterations: 110   converged: True   wall: 204.2s
beta_true: [0.8 0.5 1.  0.3 0.4]
beta_hat:  [0.7971269  0.50530734 1.20368546 0.25344274 0.46988229]
gamma_true: 0.500   gamma_hat: 0.410
alpha FE correlation: 0.892


In [7]:
{
    "objective": fit.objective,
    "runtime_seconds": fit.runtime_seconds,
    "active_cuts": fit.n_active_cuts,
}

{'objective': 293127.87655090465,
 'runtime_seconds': 204.1994086659979,
 'active_cuts': 11050}

## References

Boros, Endre, and Peter L. Hammer. 2002. "Pseudo-Boolean Optimization." *Discrete Applied Mathematics* 123(1-3): 155-225.